# AutoGaze Master Inference Suite

This notebook serves as the central hub for testing and benchmarking all AutoGaze model combinations (SigLIP, NVILA, V-JEPA2, Qwen-VL).

---

## 1. Setup Environment


In [ ]:
import os, time, torch
import numpy as np
from pathlib import Path
from PIL import Image
from autogaze.eval.models import load_runner, RUNNERS

# Configure Paths
REPO_ROOT = Path(os.getcwd()).parent
AG_PATH = str(REPO_ROOT / "weights/AutoGaze")
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

print(f"Device: {DEVICE}")
print(f"Available Runners: {sorted(RUNNERS.keys())}")

## 2. Global Test Configuration
Select your video and question here.


In [ ]:
VIDEO_PATH = REPO_ROOT / "assets/example_input.mp4"
QUESTION   = "What is happening in this video?"
RATIO      = 0.25  # Only 25% of tokens kept

print(f"Testing on: {VIDEO_PATH.name}")

## 3. Compare Different Backbones

We will load models in **Full Integration mode** to see the actual speedup.


In [ ]:
def run_benchmark(runner_key, model_path, use_ag=True):
    print(f"\n--- Benchmarking: {runner_key} (AG={'ON' if use_ag else 'OFF'}) ---")
    
    ag_p = AG_PATH if use_ag else None
    
    # Load Runner
    runner = load_runner(
        mllm=runner_key,
        model_path=str(model_path),
        autogaze_path=ag_p,
        gazing_ratio=RATIO
    )
    
    # Simple Video Loader (16 frames)
    import cv2
    cap = cv2.VideoCapture(str(VIDEO_PATH))
    frames = []
    for _ in range(16):
        ret, frame = cap.read()
        if ret: frames.append(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
    cap.release()
    
    # Warmup
    _ = runner.run(frames, QUESTION)
    
    # Timed Run
    t0 = time.perf_counter()
    answer = runner.run(frames, QUESTION)
    elapsed = time.perf_counter() - t0
    
    print(f"Answer: {answer}")
    print(f"Latency: {elapsed*1000:.1f} ms")
    return elapsed

# Example: NVILA Native
# time_nvila = run_benchmark("nvila", REPO_ROOT / "weights/NVILA-8B-HD-Video")


## 4. Visualizing Gaze Maps


In [ ]:
# Helper for visualization
import matplotlib.pyplot as plt
import torch.nn.functional as F

def visualize_runner_gaze(runner, frames):
    # This requires the runner to have _run_autogaze (most do)
    if not hasattr(runner, '_run_autogaze'):
        print("Runner does not support direct gaze visualization.")
        return
        
    gaze_map = runner._run_autogaze(frames)[0].cpu().float().numpy()
    T = min(len(frames), 8)
    fig, axes = plt.subplots(1, T, figsize=(T*3, 3))
    for t in range(T):
        axes[t].imshow(frames[t])
        axes[t].imshow(gaze_map[t], cmap='jet', alpha=0.4, extent=[0, 224, 224, 0])
        axes[t].axis('off')
    plt.show()

# Example usage:
# visualize_runner_gaze(runner, frames)